In [ ]:
import mdtraj as md
import numpy as np
from scipy.spatial.distance import pdist
import parmed as pmd
from warnings import filterwarnings
filterwarnings('ignore')
import logging
import os
# -------------------- Loading the trajectory and topology files --------------------

input_path = f'/avogadro/joe/pure_tip4p_sim/300K/'
dx_file = '/gibbs/arghavan/hp-results-pc/pairwise_energy_results/bulk_tip4p2005/pairwise_E_tip4p2005.dx'
prob_file = '/gibbs/arghavan/hp-results-pc/pairwise_energy_results/bulk_tip4p2005/prob_tip4p2005.dat'
numDens_file = '/gibbs/arghavan/hp-results-pc/pairwise_energy_results/bulk_tip4p2005/numDens_tip4p2005.dat'
logfile = '/gibbs/arghavan/hp-results-pc/pairwise_energy_results/bulk_tip4p2005/process_tip4p2005.log'



traj_path = f'{input_path}md.nc'
top_path = f'{input_path}topol.prmtop'

traj = md.load(traj_path, top=top_path)
tpl = traj.top
parm = pmd.load_file(top_path)

all_ox_tpl_indices = tpl.select("name O")
all_wat_atm_tpl_indices = tpl.select("water")
n_frames = traj.n_frames

In [ ]:
import os
import pickle
import logging
import numpy as np
from scipy.spatial.distance import pdist

# Assumes these are defined upstream:
#   parm                 - ParmEd AmberParm
#   traj                 - MDTraj trajectory (xyz in nm)
#   n_frames             - number of frames to process
#   all_ox_tpl_indices   - np.array of oxygen atom indices (first atom of each TIP4P water)
#   logfile, dx_file     - output paths
#   prob_file, numDens_file - histogram output paths
#   mean_nbr_path        - pickle with the bulk mean number of neighbours (same format as plates)
# TIP4P atom order per water: O, H1, H2, EP (M-site)

# --------------------------- Logging Setup ---------------------------

os.makedirs(os.path.dirname(logfile), exist_ok=True)

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.handlers = []

file_handler = FlushFileHandler(logfile, mode='w')
file_handler.setFormatter(logging.Formatter('%(asctime)s %(message)s', datefmt='%I:%M:%S %p'))
logger.addHandler(file_handler)

# --------------------------- Constants ---------------------------

# O-O Lennard-Jones coefficients (kcal/mol * A^12 and kcal/mol * A^6)
Acoeff = float(parm.parm_data['LENNARD_JONES_ACOEF'][0])
Bcoeff = float(parm.parm_data['LENNARD_JONES_BCOEF'][0])

# Charges in Amber internal units so that q_i*q_j/r(A) gives kcal/mol
chg_lst = np.array(parm.parm_data['CHARGE'][:4], dtype=float) * 18.2223
h_charge = chg_lst[1]   # H1 / H2
m_charge = chg_lst[3]   # EP (M-site); TIP4P oxygen carries no charge

# Charges of the charged sites only: H1, H2, M
site_charges = np.array([h_charge, h_charge, m_charge])
qq = np.outer(site_charges, site_charges)          # (3, 3) charge products

radius = 3.5                          # O-O cutoff, Angstrom
coord_min, coord_max = 0.80, 4.00     # interior sub-box, nm (avoids PBC / wrapped molecules)

# --------------------------- Main Loop ---------------------------

E_nbr_ww = []

for frame_num in range(n_frames):

    logger.info(f"Processing frame {frame_num + 1}")

    # ---------------- Select waters whose oxygen lies in the interior sub-box ----------------
    ox_coords = traj.xyz[frame_num][all_ox_tpl_indices]
    mask = np.all((ox_coords >= coord_min) & (ox_coords <= coord_max), axis=1)
    selected_ox = all_ox_tpl_indices[mask]
    n_wat = len(selected_ox)

    if n_wat < 2:
        logger.info(f"  Frame {frame_num + 1}: fewer than 2 waters selected, skipping")
        continue

    # Coordinates of all 4 sites of each selected water, in Angstrom: shape (n_wat, 4, 3)
    atom_idx = selected_ox[:, None] + np.arange(4)[None, :]
    wat_coords = traj.xyz[frame_num][atom_idx] * 10.0

    # ---------------- O-O distances and cutoff ----------------
    oo_dist = pdist(wat_coords[:, 0, :])                 # condensed, i < j
    ii, jj = np.triu_indices(n_wat, k=1)                 # same ordering as pdist
    within = oo_dist <= radius

    ii, jj, r_oo = ii[within], jj[within], oo_dist[within]
    if len(r_oo) == 0:
        continue

    # ---------------- Lennard-Jones (O-O only) ----------------
    lj_energy = Acoeff / r_oo**12 - Bcoeff / r_oo**6

    # ---------------- Electrostatics (H1, H2, M on each molecule) ----------------
    sites_i = wat_coords[ii, 1:, :]                      # (n_pairs, 3, 3)
    sites_j = wat_coords[jj, 1:, :]
    diff = sites_i[:, :, None, :] - sites_j[:, None, :, :]   # (n_pairs, 3, 3, 3)
    r_sites = np.linalg.norm(diff, axis=-1)                  # (n_pairs, 3, 3)
    elec_energy = np.sum(qq / r_sites, axis=(1, 2))

    # Half of each pair energy assigned to each molecule
    total_energy = (lj_energy + elec_energy) / 2.0
    E_nbr_ww.append(total_energy)

    logger.info(f"  {n_wat} waters selected, {len(total_energy)} pairs within {radius} A")

E_nbr_ww = np.concatenate(E_nbr_ww) if E_nbr_ww else np.array([])

# --------------------------- Save ---------------------------

header = (f"Pairwise energy values for {n_frames} frames of TIP4P\n"
          f"Column: E_n (kcal/mol), half of each pair energy, O-O cutoff {radius} A")
np.savetxt(dx_file, E_nbr_ww, fmt="%.6f", header=header)
logger.info(f"Saved in {dx_file}")
logger.info(f"{len(E_nbr_ww)} pairwise energies calculated.")

# --------------------------- Data for plotting ---------------------------


N_nbr = 5.09

bin_width = 0.06  # kcal/mol
bins = np.arange(np.floor(np.min(E_nbr_ww) / bin_width) * bin_width,
                 np.ceil(np.max(E_nbr_ww) / bin_width) * bin_width + bin_width, bin_width)

counts, edges = np.histogram(E_nbr_ww, bins=bins)
bin_centers = edges[:-1] + bin_width / 2

prob = counts / (bin_width * np.sum(counts))
rho_i = prob * N_nbr

np.savetxt(prob_file, np.column_stack([bin_centers, prob]), fmt=["%.4f", "%.6f"])
np.savetxt(numDens_file, np.column_stack([bin_centers, rho_i]), fmt=["%.4f", "%.6f"])
logger.info(f"Saved {prob_file} and {numDens_file} (N_nbr = {N_nbr})")

# save for plotting